# 🦠 MicroNet — Phase 2: Interaction Inference (Colab)

This notebook performs:
1. **SPIEC-EASI** — Sparse inverse covariance network inference (Python-native GraphicalLassoCV)
2. **gLV** — Generalized Lotka-Volterra interaction matrix inference (LASSO/Ridge)

Upload your `clr_abundance_matrix.tsv` and (optionally) `metadata.tsv` to Colab or mount Google Drive.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.covariance import GraphicalLassoCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (optional — skip if uploading files manually)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit these paths to match your setup
# ══════════════════════════════════════════════════════════════════════════

# If using Google Drive:
# ABUNDANCE_PATH = '/content/drive/MyDrive/MicroNet/results/profiling/clr_abundance_matrix.tsv'
# METADATA_PATH  = '/content/drive/MyDrive/MicroNet/data/hmp/metadata.tsv'
# OUTDIR         = '/content/drive/MyDrive/MicroNet/results/inference'

# If uploading files directly to Colab:
ABUNDANCE_PATH = 'clr_abundance_matrix.tsv'
METADATA_PATH  = 'metadata.tsv'  # Optional: needed only for time-series gLV
OUTDIR         = 'results/inference'

GLV_METHOD    = 'lasso'   # 'lasso' or 'ridge'
GLV_THRESHOLD = 0.01     # Interaction classification threshold

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUTDIR}')

In [ ]:
# ── Load CLR abundance matrix ─────────────────────────────────────────────
clr = pd.read_csv(ABUNDANCE_PATH, sep='\t', index_col=0)
taxa = clr.columns.tolist()
print(f'Loaded: {clr.shape[0]} samples × {clr.shape[1]} taxa')
clr.head()

---
## Phase 2A: SPIEC-EASI (Python-native)

Uses `sklearn.covariance.GraphicalLassoCV` — the same graphical LASSO algorithm
that SPIEC-EASI uses with `method="glasso"`. The precision matrix Θ gives
partial correlations: `ρ_ij = -Θ_ij / sqrt(Θ_ii × Θ_jj)`.

In [ ]:
def run_graphical_lasso(clr_df, n_lambdas=20, cv_folds=5):
    """
    Python-native SPIEC-EASI equivalent using GraphicalLassoCV.
    Input: CLR-transformed abundance matrix (samples × taxa).
    Returns: weighted adjacency (partial correlations), binary adjacency.
    """
    X = clr_df.values.astype(np.float64)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Graphical LASSO with cross-validated regularization
    alphas = np.logspace(-2, 0, n_lambdas)
    model = GraphicalLassoCV(
        alphas=alphas,
        cv=min(cv_folds, X.shape[0]),
        max_iter=500,
        assume_centered=False,
    )
    model.fit(X_scaled)
    print(f'Selected alpha (regularization): {model.alpha_:.4f}')

    # Precision matrix → partial correlations
    precision = model.precision_
    diag = np.sqrt(np.diag(precision))
    diag[diag == 0] = 1e-10
    partial_corr = -precision / np.outer(diag, diag)
    np.fill_diagonal(partial_corr, 0)

    # Binary adjacency: edges where precision is non-zero
    binary = (np.abs(precision) > 1e-8).astype(int)
    np.fill_diagonal(binary, 0)

    # Mask: only keep selected edges (same logic as SPIEC-EASI adj_masked)
    adj_masked = partial_corr * binary

    taxa_names = clr_df.columns.tolist()
    adj_df = pd.DataFrame(adj_masked, index=taxa_names, columns=taxa_names)
    bin_df = pd.DataFrame(binary, index=taxa_names, columns=taxa_names)

    n_edges = int(binary.sum()) // 2
    density = n_edges / (len(taxa_names) * (len(taxa_names) - 1) / 2)
    print(f'Network: {len(taxa_names)} nodes, {n_edges} edges, density={density:.4f}')

    return adj_df, bin_df, model

print('SPIEC-EASI functions defined ✓')

In [ ]:
# ── Run SPIEC-EASI ────────────────────────────────────────────────────────
print('Running GraphicalLassoCV (SPIEC-EASI equivalent) ...')
adj_weighted, adj_binary, glasso_model = run_graphical_lasso(clr)

# Save weighted adjacency (masked — only selected edges)
adj_weighted.to_csv(f'{OUTDIR}/spieceasi_adj_weighted.tsv', sep='\t')

# Build edge list
edges = []
for i, t1 in enumerate(taxa):
    for j, t2 in enumerate(taxa[i+1:], i+1):
        w = adj_weighted.iloc[i, j]
        if w != 0:
            edges.append({
                'from': t1, 'to': t2,
                'weight': w, 'abs_weight': abs(w),
                'sign': 'positive' if w > 0 else 'negative'
            })
edge_df = pd.DataFrame(edges).sort_values('abs_weight', ascending=False)
edge_df.to_csv(f'{OUTDIR}/spieceasi_edgelist.tsv', sep='\t', index=False)

print(f'\nTop 10 edges by strength:')
edge_df.head(10)

In [ ]:
# ── Network stats ────────────────────────────────────────────────────────
pos_edges = edge_df[edge_df['sign'] == 'positive']
neg_edges = edge_df[edge_df['sign'] == 'negative']

stats_dict = {
    'n_nodes': len(taxa),
    'n_edges': len(edge_df),
    'n_positive': len(pos_edges),
    'n_negative': len(neg_edges),
    'density': len(edge_df) / (len(taxa) * (len(taxa)-1) / 2),
    'mean_abs_weight': edge_df['abs_weight'].mean(),
}
stats_df = pd.DataFrame.from_dict(stats_dict, orient='index', columns=['value'])
stats_df.to_csv(f'{OUTDIR}/spieceasi_network_stats.tsv', sep='\t')

print('SPIEC-EASI complete ✓')
print(stats_df.to_string())

---
## Phase 2B: gLV Interaction Inference

Generalized Lotka-Volterra model: `dx_i/dt = x_i × (r_i + Σ_j A_ij × x_j)`

- **Time-series**: linearize to `(1/x_i) dx_i/dt = r_i + A_i @ X` and solve via LASSO
- **Cross-sectional**: pseudo-steady-state LIMITS-style regression (Fisher & Mehta 2014)

In [ ]:
# ── gLV Core Functions ─────────────────────────────────────────────────────

def compute_derivatives(X, times):
    """Finite-difference derivatives for time-series data."""
    dX_dt = np.diff(X, axis=0) / np.diff(times)[:, None]
    X_mid = (X[:-1] + X[1:]) / 2
    return dX_dt, X_mid


def fit_glv_lasso(X, dX_dt, alpha_range=None, cv=5):
    """Fit gLV interaction matrix A via per-taxon LASSO regression."""
    N = X.shape[1]
    assert X.shape == dX_dt.shape, f'Shape mismatch: X {X.shape} vs dX_dt {dX_dt.shape}'

    X_safe = np.where(X > 1e-10, X, 1e-10)
    percap_growth = dX_dt / X_safe

    A = np.zeros((N, N))
    r = np.zeros(N)
    if alpha_range is None:
        alpha_range = np.logspace(-4, 1, 50)

    for i in range(N):
        model = LassoCV(alphas=alpha_range, cv=cv, max_iter=10000, fit_intercept=True)
        model.fit(X, percap_growth[:, i])
        r[i] = model.intercept_
        A[i, :] = model.coef_
    return A, r


def fit_glv_ridge(X, dX_dt):
    """Fit gLV via Ridge — more stable when N > samples."""
    N = X.shape[1]
    assert X.shape == dX_dt.shape, f'Shape mismatch: X {X.shape} vs dX_dt {dX_dt.shape}'

    X_safe = np.where(X > 1e-10, X, 1e-10)
    percap_growth = dX_dt / X_safe
    A = np.zeros((N, N))
    r = np.zeros(N)

    model = RidgeCV(alphas=np.logspace(-3, 3, 50), fit_intercept=True)
    for i in range(N):
        model.fit(X, percap_growth[:, i])
        r[i] = model.intercept_
        A[i, :] = model.coef_
    return A, r


def fit_glv_pss(X, alpha_range=None, method='lasso'):
    """
    Pseudo-steady-state (LIMITS-style) inference for cross-sectional data.
    At equilibrium: dx/dt ≈ 0, so x_i ≈ f(x_1, ..., x_N).
    Ref: Fisher & Mehta (2014), PLOS ONE.
    """
    N, S = X.shape[1], X.shape[0]
    A = np.zeros((N, N))
    r = np.zeros(N)
    if alpha_range is None:
        alpha_range = np.logspace(-4, 1, 50)

    for i in range(N):
        y = X[:, i]
        X_others = np.delete(X, i, axis=1)
        if method == 'lasso':
            model = LassoCV(alphas=alpha_range, cv=min(5, S), max_iter=10000,
                            fit_intercept=True)
        else:
            model = RidgeCV(alphas=np.logspace(-3, 3, 50), fit_intercept=True)
        model.fit(X_others, y)
        r[i] = model.intercept_
        coefs = model.coef_
        A[i, :i]   = coefs[:i]
        A[i, i+1:] = coefs[i:]
        A[i, i] = -np.abs(coefs).mean()  # Heuristic self-regulation
    return A, r


def classify_interactions(A, threshold=0.01):
    """Classify pairwise interactions from the A matrix."""
    N = A.shape[0]
    records = []
    type_map = {
        '+/+': 'Mutualism',  '-/-': 'Competition',
        '+/-': 'Parasitism', '-/+': 'Parasitism',
        '+/0': 'Commensalism', '0/+': 'Commensalism',
        '-/0': 'Amensalism',   '0/-': 'Amensalism',
    }
    for i in range(N):
        for j in range(i+1, N):
            a_ij, a_ji = A[i,j], A[j,i]
            if abs(a_ij) < threshold and abs(a_ji) < threshold:
                continue
            strength = (abs(a_ij) + abs(a_ji)) / 2
            if strength < threshold:
                continue
            s_ij = '+' if a_ij > threshold else ('-' if a_ij < -threshold else '0')
            s_ji = '+' if a_ji > threshold else ('-' if a_ji < -threshold else '0')
            sig = f'{s_ij}/{s_ji}'
            records.append({
                'taxon_i': i, 'taxon_j': j,
                'A_ij': a_ij, 'A_ji': a_ji,
                'signature': sig,
                'interaction_type': type_map.get(sig, 'Unknown'),
                'strength': strength,
            })
    return pd.DataFrame(records)


def community_stability(A, x_eq):
    """Compute Jacobian eigenvalues for local stability assessment."""
    N = len(x_eq)
    J = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            J[i, j] = x_eq[i] * A[i, j]
    eigenvalues = np.linalg.eigvals(J)
    max_real = np.max(np.real(eigenvalues))
    return {
        'eigenvalues': eigenvalues,
        'is_stable': max_real < 0,
        'resilience': -max_real,
        'reactivity': np.max(np.real(np.linalg.eigvals((J + J.T) / 2))),
    }

print('gLV functions defined ✓')

In [ ]:
# ── Run gLV Inference ──────────────────────────────────────────────────────

# Detect time-series vs cross-sectional
has_timeseries = False
try:
    meta = pd.read_csv(METADATA_PATH, sep='\t', index_col=0)
    has_timeseries = 'time_point' in meta.columns and 'subject_id' in meta.columns
except FileNotFoundError:
    meta = None
    print('No metadata file found — using cross-sectional (pseudo-steady-state) mode')

if has_timeseries:
    print('Time-series data detected — using temporal gLV')
    all_X, all_dX = [], []
    for subj, grp in meta.groupby('subject_id'):
        grp = grp.sort_values('time_point')
        if len(grp) < 2:
            print(f'  Skipping subject {subj}: only {len(grp)} timepoint')
            continue
        X = clr.loc[grp.index].values
        times = grp['time_point'].values.astype(float)
        dX, X_mid = compute_derivatives(X, times)
        all_X.append(X_mid)
        all_dX.append(dX)
    if all_X:
        X_all = np.vstack(all_X)
        dX_all = np.vstack(all_dX)
    else:
        print('No subjects with >=2 timepoints. Falling back to PSS.')
        has_timeseries = False

if not has_timeseries:
    print(f'Cross-sectional data — pseudo-steady-state (LIMITS-style)')
    X_all = clr.values
    dX_all = None

# Fit
print(f'\nFitting gLV via {GLV_METHOD.upper()} ...')
if dX_all is None:
    A, r = fit_glv_pss(X_all, method=GLV_METHOD)
elif GLV_METHOD == 'lasso':
    A, r = fit_glv_lasso(X_all, dX_all)
else:
    A, r = fit_glv_ridge(X_all, dX_all)

print(f'Interaction matrix shape: {A.shape}')
print(f'Non-zero entries: {np.count_nonzero(A)} / {A.size}')

In [ ]:
# ── Save & classify interactions ───────────────────────────────────────────
A_df = pd.DataFrame(A, index=taxa, columns=taxa)
A_df.to_csv(f'{OUTDIR}/glv_interaction_matrix.tsv', sep='\t')

interactions = classify_interactions(A, threshold=GLV_THRESHOLD)
interactions['taxon_i_name'] = interactions['taxon_i'].map(lambda i: taxa[i])
interactions['taxon_j_name'] = interactions['taxon_j'].map(lambda i: taxa[i])
interactions.to_csv(f'{OUTDIR}/classified_interactions.tsv', sep='\t', index=False)

print(f'Classified {len(interactions)} interactions:')
print(interactions['interaction_type'].value_counts())

In [ ]:
# ── Stability analysis ─────────────────────────────────────────────────────
clr_means = clr.mean().values
raw_props = np.exp(clr_means)
x_eq = raw_props / raw_props.sum()

stab = community_stability(A, x_eq)
print(f"Community stability: {'STABLE ✅' if stab['is_stable'] else 'UNSTABLE ❌'}")
print(f"  Resilience:  {stab['resilience']:.4f}")
print(f"  Reactivity:  {stab['reactivity']:.4f}")

In [ ]:
# ── Visualizations ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Interaction matrix heatmap (top 30 taxa)
n_show = min(30, len(taxa))
vmax = np.percentile(np.abs(A[:n_show, :n_show]), 95)
sns.heatmap(A[:n_show, :n_show],
            xticklabels=taxa[:n_show], yticklabels=taxa[:n_show],
            cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
            ax=axes[0], square=True, linewidths=0.3)
axes[0].set_title('gLV Interaction Matrix (A_ij: effect of j on i)', fontsize=12)
axes[0].tick_params(labelsize=6)

# Interaction type bar chart
if len(interactions) > 0:
    colors = {'Mutualism': '#2ECC71', 'Competition': '#E74C3C',
              'Parasitism': '#E67E22', 'Commensalism': '#3498DB',
              'Amensalism': '#9B59B6', 'Unknown': '#BDC3C7'}
    counts = interactions['interaction_type'].value_counts()
    counts.plot.bar(ax=axes[1], color=[colors.get(t, '#999') for t in counts.index])
    axes[1].set_title('Interaction Type Distribution', fontsize=12)
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{OUTDIR}/inference_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ All inference outputs saved to {OUTDIR}/')